In [9]:
import os, re, json, math, time, random
from typing import List, Dict, Any, Optional, Tuple
import numpy as np, pandas as pd
import requests
from IPython.display import Markdown, display, update_display
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
from datasets import load_dataset, load_from_disk
import torch
import gradio as gr
from textwrap import dedent
import gc



In [8]:
torch.cuda.empty_cache()
gc.collect()

18957

In [2]:
env_path = find_dotenv(".env", usecwd=True)
load_dotenv(env_path, override=True)
HF_TOKEN = (os.getenv("HUGGINGFACE_HUB_TOKEN"))

In [3]:
LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"

In [ ]:
torch.cuda.empty_cache()
gc.collect()

print("🔄 Cargando modelo Llama...")
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
      LLAMA,
      device_map="auto",
      quantization_config=quant_config,
      offload_buffers=True
  )
print("✅ Modelo cargado exitosamente")

🔄 Cargando modelo Llama...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Modelo cargado exitosamente


In [10]:
def generar_ofertas_por_pais(pais_seleccionado, progress=gr.Progress()):
      """
      Genera ofertas salariales para un país específico usando el modelo Llama
      """
      progress(0, desc="Preparando configuración...")

      # Mapeo de países a monedas y ciudades
      paises_config = {
          "Chile": {"moneda": "CLP", "ciudad": "Santiago", "salario_minimo": 460000},
          "Argentina": {"moneda": "ARS", "ciudad": "Buenos Aires", "salario_minimo": 156000},
          "México": {"moneda": "MXN", "ciudad": "Ciudad de México", "salario_minimo": 6398},
          "Colombia": {"moneda": "COP", "ciudad": "Bogotá", "salario_minimo": 1300000},
          "Perú": {"moneda": "PEN", "ciudad": "Lima", "salario_minimo": 1025},
          "España": {"moneda": "EUR", "ciudad": "Madrid", "salario_minimo": 1134},
          "Estados Unidos": {"moneda": "USD", "ciudad": "Miami", "salario_minimo": 2080}
      }

      config = paises_config.get(pais_seleccionado, paises_config["Chile"])

      progress(0.1, desc="Preparando prompt...")

      # System message
      system_message = (
          "Eres un analista de compensación para una cadena de supermercados. "
          "Debes emitir únicamente JSON VÁLIDO conforme al esquema del usuario. "
          "No incluyas explicaciones ni texto adicional. Evita PII y atributos protegidos. "
          "Debes considerar las tendencias actuales del mercado laboral. Realistas y conformes a la industria. "
          "Todas las respuestas deben estar en español."
      )

      # User prompt
      user_prompt_pais = dedent(f"""\
          A continuación te dejo los títulos de los puestos a los cuales se les debe generar una oferta salarial:
          Cajero, Repositor, Carnicero, Charcutero, Panadero, Pescadero, Supervisor de Caja, Supervisor de Piso, Gerente de Tienda, Supervisor Prevención de Pérdidas.
          
          IMPORTANTE: Todos los datos deben ser para {pais_seleccionado}, usando la moneda {config['moneda']} y la ciudad {config['ciudad']}.
          
          Debes responder ÚNICAMENTE con un JSON por cada cargo (sin texto adicional). A continuación un ejemplo de la estructura esperada (plantilla):
          El archivo JSON final no pueden contener formulas, expresiones matemáticas, ni texto adicional. Solo números ya calculados.
                        
           {{
            "empresa": "SuperMercado {pais_seleccionado}",
            "familia_cargo": "{{familia_cargo}}",
            "cargo": "{{cargo}}",
            "nivel": "{{nivel}}",
            "tienda_tier": "{{tienda_tier}}",
            "ubicacion": {{
              "pais": "{pais_seleccionado}",
              "ciudad": "{config['ciudad']}",
              "indice_costo_vida": {{indice_costo_vida}}
            }},
            "moneda": "{config['moneda']}",
            "horas_semanales": {{horas_semanales}},
            "tipo_contrato": "{{tipo_contrato}}",
            "turno": "{{turno}}",
            "salario_base_mensual": {{salario_base_mensual}},
            "salario_base_hora": {{salario_base_hora}},
            "porc_bono_variable": {{porc_bono_variable}},
            "asignaciones": {{
              "transporte": {{transporte}},
              "tarjeta_alimentacion": {{tarjeta_alimentacion}},
              "asistencia": {{asistencia}},
              "porc_diferencial_nocturno": {{porc_diferencial_nocturno}}
            }},
            "beneficios": {{beneficios}},
            "fecha_propuesta": "{{fecha_propuesta}}",
            "justificacion": "{{justificacion}}",
            "estimados_usd": {{
              "base_usd": {{base_usd}},
              "asignaciones_usd": {{asignaciones_usd}},
              "total_efectivo_usd": {{total_efectivo_usd}}
            }},
            "restricciones": {{
              "salario_minimo_mensual": {config['salario_minimo']}
            }}
          }}
                               
          REGLAS DURAS Y CONSISTENCIA:
          - Devuelve un ARREGLO JSON con 10 objetos (uno por cada cargo en el listado, en el mismo orden).
          - Usa números (no strings) para cantidades y porcentajes.
          - IMPORTANTE: Todos los valores numéricos deben ser números calculados, NO fórmulas ni expresiones matemáticas.
          - Los salarios deben ser realistas para {pais_seleccionado} y estar en {config['moneda']}.
          - Calcula "salario_base_hora" dividiendo salario_base_mensual entre (4.3333 * horas_semanales), redondeado a 2 decimales.
          - Si "turno" != "nocturno", entonces asignaciones.porc_diferencial_nocturno = 0
          - Para "estimados_usd.base_usd": multiplica salario_base_mensual por tipo de cambio (usa 1.0 si no aplica).
          - Para "estimados_usd.asignaciones_usd": suma todas las asignaciones (transporte + tarjeta_alimentacion + asistencia + diferencial nocturno si aplica) y 
            multiplica por tipo de cambio.
          - Para "estimados_usd.total_efectivo_usd": suma base_usd + asignaciones_usd + bono variable calculado.
          - Prohíbese cualquier PII o atributo protegido.
          - Salida: solo el arreglo JSON válido, sin texto adicional, con todos los valores ya calculados como números.
          
          """)

      progress(0.3, desc="Tokenizando...")

      messages_pais = [
          {"role": "system", "content": system_message},
          {"role": "user", "content": user_prompt_pais}
      ]

      inputs_pais = tokenizer.apply_chat_template(messages_pais, return_tensors="pt").to("cuda")

      progress(0.5, desc="Generando ofertas con Llama...")

      # Generar sin streamer para capturar output
      outputs_pais = model.generate(
          inputs_pais,
          max_new_tokens=10000,
          temperature=0.7,
          top_p=0.9,
          do_sample=True
      )

      progress(0.8, desc="Procesando respuesta...")

      response = tokenizer.decode(outputs_pais[0], skip_special_tokens=True)

      # Extraer el JSON de la respuesta
      json_match = re.search(r'\[[\s\S]*\]', response)

      if json_match:
          json_str = json_match.group()
          ofertas_salariales = json.loads(json_str)

          progress(1.0, desc="¡Completado!")

          # Crear tabla resumen
          tabla_md = crear_tabla_resumen(ofertas_salariales)

          # Crear detalles
          detalle_md = crear_detalle_ofertas(ofertas_salariales)

          # JSON formateado
          json_formateado = json.dumps(ofertas_salariales, indent=2, ensure_ascii=False)

          return tabla_md, detalle_md, json_formateado
      else:
          return "⚠ No se encontró JSON en la respuesta", "", response

In [11]:
def crear_tabla_resumen(ofertas):
      """Crea tabla resumen en Markdown"""
      if not ofertas:
          return "No hay datos para mostrar"

      tabla = "| Cargo | Nivel | Ubicación | Salario Base | Bono | Turno | Tier |\n"
      tabla += "|---|---|---|---:|---:|---|---|\n"

      for oferta in ofertas:
          cargo = oferta.get("cargo", "—")
          nivel = oferta.get("nivel", "—")
          moneda = oferta.get("moneda", "CLP")

          ubic = oferta.get("ubicacion", {})
          ciudad = ubic.get("ciudad", "—")
          pais = ubic.get("pais", "—")
          ubicacion = f"{ciudad}, {pais}"

          base = oferta.get("salario_base_mensual", 0)
          base_fmt = f"{moneda} {base:,.0f}".replace(",", ".")

          bono = oferta.get("porc_bono_variable", 0) * 100
          bono_fmt = f"{bono:.0f}%"

          turno = oferta.get("turno", "—")
          tier = oferta.get("tienda_tier", "—")

          tabla += f"| {cargo} | {nivel} | {ubicacion} | {base_fmt} | {bono_fmt} | {turno} | {tier} |\n"

      return tabla

def crear_detalle_ofertas(ofertas_salariales):
      """Crea detalle completo de ofertas en Markdown"""
      detalle_completo = "## 📋 Detalle por Cargo\n\n"

      for i, oferta in enumerate(ofertas_salariales, 1):
          cargo = oferta.get("cargo", "—")
          nivel = oferta.get("nivel", "—")
          moneda = oferta.get("moneda", "CLP")

          # Información básica
          md = f"### {i}. {cargo} ({nivel})\n\n"

          # Ubicación
          ubic = oferta.get("ubicacion", {})
          md += f"**📍 Ubicación:** {ubic.get('ciudad', '—')}, {ubic.get('pais', '—')}\n\n"

          # Compensación
          base = oferta.get("salario_base_mensual", 0)
          hora = oferta.get("salario_base_hora", 0)
          bono = oferta.get("porc_bono_variable", 0) * 100

          md += f"**💰 Compensación:**\n"
          md += f"- Salario base mensual: {moneda} {base:,.0f}\n".replace(",", ".")
          md += f"- Salario por hora: {moneda} {hora:,.2f}\n".replace(",", ".")
          md += f"- Bono variable: {bono:.0f}%\n\n"

          # Asignaciones
          asig = oferta.get("asignaciones", {})
          md += f"**🎁 Asignaciones:**\n"
          if asig.get("transporte", 0) > 0:
              md += f"- Transporte: {moneda} {asig['transporte']:,.0f}\n".replace(",", ".")
          if asig.get("tarjeta_alimentacion", 0) > 0:
              md += f"- Alimentación: {moneda} {asig['tarjeta_alimentacion']:,.0f}\n".replace(",", ".")
          if asig.get("asistencia", 0) > 0:
              md += f"- Asistencia: {moneda} {asig['asistencia']:,.0f}\n".replace(",", ".")
          if asig.get("porc_diferencial_nocturno", 0) > 0:
              md += f"- Diferencial nocturno: {asig['porc_diferencial_nocturno']*100:.0f}%\n"
          md += "\n"

          # Condiciones
          md += f"**📝 Condiciones:**\n"
          md += f"- Tipo contrato: {oferta.get('tipo_contrato', '—')}\n"
          md += f"- Horas semanales: {oferta.get('horas_semanales', '—')}\n"
          md += f"- Turno: {oferta.get('turno', '—')}\n"
          md += f"- Tier tienda: {oferta.get('tienda_tier', '—')}\n\n"

          # Justificación
          just = oferta.get("justificacion", "")
          if just:
              md += f"**💡 Justificación:** {just}\n\n"

          md += "---\n\n"

          detalle_completo += md

      return detalle_completo

In [ ]:
with gr.Blocks(theme=gr.themes.Soft(), title="Generador de Ofertas Salariales") as demo:

      gr.Markdown("""
      # 🛒 Generador de Ofertas Salariales - Supermercados
      ### Generación de datos sintéticos usando Llama 3.1 8B
      
      Selecciona un país y genera ofertas salariales realistas para 10 cargos de supermercado.
      """)

      with gr.Row():
          with gr.Column(scale=1):
              pais_dropdown = gr.Dropdown(
                  choices=[
                      "Chile",
                      "Argentina",
                      "México",
                      "Colombia",
                      "Perú",
                      "España",
                      "Estados Unidos"
                  ],
                  label="🌎 Selecciona el País",
                  value="Chile",
                  info="Elige el país para generar las ofertas salariales"
              )

              generar_btn = gr.Button(
                  "🚀 Generar Ofertas Salariales",
                  variant="primary",
                  size="lg"
              )

              gr.Markdown("""
              ### 📋 Cargos incluidos:
              1. Cajero
              2. Repositor
              3. Carnicero
              4. Charcutero
              5. Panadero
              6. Pescadero
              7. Supervisor de Caja
              8. Supervisor de Piso
              9. Gerente de Tienda
              10. Supervisor Prevención de Pérdidas
              """)

      with gr.Row():
          with gr.Column():
              gr.Markdown("## 📊 Resumen de Ofertas")
              tabla_output = gr.Markdown(label="Tabla Resumen")

      with gr.Row():
          with gr.Column():
              detalle_output = gr.Markdown(label="Detalle Completo")

      with gr.Row():
          with gr.Column():
              gr.Markdown("## 💾 JSON Generado")
              json_output = gr.Code(
                  label="Datos en formato JSON",
                  language="json",
                  lines=15
              )

      # Conectar botón con función
      generar_btn.click(
          fn=generar_ofertas_por_pais,
          inputs=[pais_dropdown],
          outputs=[tabla_output, detalle_output, json_output]
      )

      gr.Markdown("""
      ---
      💡 **Nota:** La generación puede tomar entre 30-60 segundos dependiendo de tu GPU.
      """)

  # Lanzar interfaz con parámetros para auto-cerrado
demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
